# 🚀 Project Vynix: Full Dataset Training & Benchmark on NVIDIA T4 GPU
### *Train on 38,118 HICO-DET Images, Push Past 50% mAP, and Save Pretrained Models*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jatindeswal/Vynix/blob/main/Train_Vynix_Full_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-black?logo=github)](https://github.com/Jatindeswal/Vynix)

---

## 📌 Overview & Execution Workflow
This Google Colab notebook provides an **end-to-end execution pipeline** to:
1. **Download & Stream Dataset**: Ingests all 38,118 training and 9,658 test images of the official HICO-DET benchmark from HuggingFace.
2. **Extract 3-Stream Multi-Modal Features**: Processes images through YOLOv8-nano and frozen CLIP ViT-B/32 (Human, Object, and Union crops) plus the 8D Spatial Geometry vector.
3. **Train Adapter & Spatial MLP**: Optimizes non-parametric temperature $\beta$, residual weight $\alpha$, and the 2-layer GELU geometry network via AdamW.
4. **Evaluate on 9,658 Test Images**: Computes Full mAP (600 classes), Rare mAP (155 classes), and Non-Rare mAP (445 classes) with the Physical-Semantic Veto Gate.
5. **Save & Export Checkpoints**: Automatically syncs the trained `.pth` and `.pt` files to your Google Drive and allows 1-click browser download.


## ⚙️ Step 1: Environment & GPU Acceleration Verification
Verifies that a GPU runtime (NVIDIA Tesla T4 with 16 GB VRAM) is allocated, clones the repository, and installs dependencies.


In [ ]:
# Verify GPU
!nvidia-smi

# Clone Project Vynix
!git clone https://github.com/Jatindeswal/Vynix.git
%cd Vynix

# Install requirements
!pip install -q -r requirements.txt huggingface_hub pyarrow
print('✓ Environment and GPU acceleration ready!')


## 📥 Step 2: Automated HICO-DET Dataset Ingestion
Downloads all 13 training shards and 4 test shards directly from HuggingFace into `/content/dataset` via Google's high-speed cloud backbone (~2-3 minutes).


In [ ]:
import os
os.makedirs('/content/dataset', exist_ok=True)

# Download all 13 train shards and 4 test shards
!python download_data.py --dataset-dir /content/dataset --train-shards 13 --test-shards 4


## 💾 Step 3: Optional Google Drive Mount (Permanent Backup)
Mounting your Google Drive ensures your trained model (`vynix_full_model.pth` and `vynix_full_cache.pt`) is automatically saved to your Drive so you never lose it even if Colab disconnects.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/Vynix_Pretrained_Models"
print('✓ Google Drive connected! Models will be backed up to: MyDrive/Vynix_Pretrained_Models/')


## 🏋️ Step 4: Full Dataset 3-Stream Feature Extraction & Adapter Training
Executes `train_vynix_full.py`. Streams all 38,118 images, extracts 1536-d visual features and 8D geometry vectors, builds the non-parametric tensor cache, and optimizes adapter weights.


In [ ]:
!python train_vynix_full.py \
    --dataset-dir /content/dataset \
    --output-dir saved_models \
    --epochs 15 \
    --batch-size 256 \
    --device cuda

# Immediate Drive backup to prevent data loss if runtime disconnects
import os
if os.path.exists('/content/drive/MyDrive'):
    !cp saved_models/* /content/drive/MyDrive/Vynix_Pretrained_Models/
    print('✓ Checkpoints immediately secured in Google Drive!')


## 🧪 Step 5: Official Benchmark Evaluation (9,658 Test Images)
Natively loads the newly saved cache and model weights from `saved_models/`, applies the Physical-Semantic Geometric Veto Gate, and computes the official mAP scores.


In [ ]:
!python test_vynix_pretrained.py \
    --dataset-dir /content/dataset \
    --model-dir saved_models \
    --device cuda


## 📦 Step 6: Backup Checkpoints & 1-Click Download
Copies the trained model to your Google Drive and triggers a direct browser download to your local machine.


In [ ]:
# 1. Save to Google Drive
!cp saved_models/* "/content/drive/MyDrive/Vynix_Pretrained_Models/"
print('✓ Models successfully backed up to your Google Drive!')

# 2. Browser download helper
from google.colab import files
print('\nDownloading trained model weights to your laptop...')
files.download('saved_models/vynix_full_model.pth')
# files.download('saved_models/vynix_full_cache.pt')


## 🎨 Step 7: Interactive Demo Inference (Test Any Image!)
Runs the trained Vynix pipeline on an image to show real-time detection, interaction classification, and hallucination vetoing in action.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from vynix_prototype import run_vynix

# Run prototype demo with the Vynix logic gate
!python vynix_prototype.py

# Display visual proofs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
if os.path.exists('vynix_proof_holding.jpg'):
    ax1.imshow(Image.open('vynix_proof_holding.jpg'))
    ax1.set_title('Proof 1: Physical Contact Detected')
    ax1.axis('off')

if os.path.exists('vynix_proof_not_holding.jpg'):
    ax2.imshow(Image.open('vynix_proof_not_holding.jpg'))
    ax2.set_title('Proof 2: Spatial Hallucination VETOED')
    ax2.axis('off')

plt.tight_layout()
plt.show()
